<a href="https://colab.research.google.com/github/Rahat048/Batch-62/blob/main/Gemini_2_0_Video_%26_Audio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install --upgrade --quiet google_genai

In [10]:
!pip install --quiet gTTS

In [5]:
!pip install playsound

  Preparing metadata (setup.py) ... done
  Created wheel for playsound: filename=playsound-1.3.0-py3-none-any.whl size=7020 sha256=90e6f2d542d4fa52e596fd9af961ca56ee4bf7d3eaf9947af0fec497eac8c0ae
  Stored in directory: /root/.cache/pip/wheels/90/89/ed/2d643f4226fc8c7c9156fc28abd8051e2d2c0de37ae51ac45c
Successfully built playsound


In [6]:
from google.colab import userdata
GEMINI_API: str = userdata.get('GOOGLE_API_KEY')
if(GEMINI_API):
  print("API Key found")
else:
  print("API Key not found")

API Key found


In [7]:
from google import genai
from google.genai import Client

client: Client = genai.Client(
    api_key=GEMINI_API,
)

model: str = "gemini-2.0-flash-exp"

In [16]:
import time

def upload_video(video_file_name):
  video_file = client.files.upload(path=video_file_name)
  while video_file.state == "PROCESSING":
      print('Waiting for video to be processed.')
      time.sleep(10)
      video_file = client.files.get(name=video_file.name or "")

  if video_file.state == "FAILED":
    raise ValueError(video_file.state)
  print(f'Video processing complete: ' + (video_file.uri or ""))

  return video_file

In [17]:
my_video = upload_video(video_file_name = "/content/WhatsApp Video 2024-12-28 at 01.50.00_a4faf1df.mp4" )

Waiting for video to be processed.
Video processing complete: https://generativelanguage.googleapis.com/v1beta/files/6i9sn1v0b657


In [18]:
from google.genai.types import Content, Part
from IPython.display import display, Markdown

In [20]:
prompt = """For each scene in this video,
            generate captions that describe the scene along with any spoken text placed in quotation marks.
            Place each caption into an object with the timecode of the caption in the video.
            also stor the adult voice in the video in one text form.
         """

video = my_video
response = client.models.generate_content(
    model=model,
    contents=[
        Content(
            role="user",
            parts=[
                Part.from_uri(
                    file_uri=video.uri or "",
                    mime_type=video.mime_type or ""),
                ]),
        prompt,
    ]
)

Markdown(response.text)

```json
[
  {
    "00:00": "A person is seen walking up a set of stairs. The person's legs are visible but the rest of their body is out of frame."
  },
  {
   "00:01":"The camera moves to the left, revealing white walls, and a set of stairs going upwards."
  },
    {
   "00:02":"A child is seen standing at the top of the stairs, a loud noise is heard, and the child runs down a few steps, then is seen crawling up the stairs."
  },
  {
   "00:04": "The child is crawling up the stairs from a low camera angle. There is an adult voice saying 'bhaag'."
  },
  {
    "00:05": "The child continues to crawl up the stairs as the adult voice repeats 'bhaag'."
   },
  {
    "00:08": "The child is seen crawling up the stairs with the adult voice saying 'tu bhaag, tu bhaag ja. Mai kehta hu tu bhaag, bhaag ja.' "
  },
  {
    "00:11": "The child reaches the bottom of the stairs and is seen walking around the room."
  },
  {
     "adultVoice":"bhaag tu bhaag tu bhaag ja mai kehta hu tu bhaag bhaag ja"
  }
]
```

In [23]:
from gtts import gTTS
from playsound import playsound
from IPython.display import Audio

In [29]:
tts = gTTS(text='bhaag tu bhaag tu bhaag ja mai kehta hu tu bhaag bhaag ja', lang='en')

with open('output.mp3', 'wb') as f:
    for chunk in tts.stream():
        f.write(chunk)

In [30]:
display(Audio('output.mp3', autoplay=True))